# Exp25 — Word order with parallel items (fix for Exp22)

Exp22 found word-order late-layer `frac_flip` at 0.03–0.19 across languages (the first task family in your design where cross-language patching does **not** transfer), but couldn't place word order on the representation axis because items were different sentences per language. This notebook rebuilds word order with **parallel scenes** — one semantic concept realized in every language — so the same captured states support (a) patching compat / `frac_flip` with same-scene opposite-order cross donors and (b) cross-language `R_retr` and CKA on decision-position states.

**Three sub-phenomena**, each with ~10–15 parallel scenes across 7 languages (EN, FR, ES, DE, AR, HI, ZH):
- **adj**: adjective-noun order. EN/DE/ZH/HI are adj-N (pre); FR/ES/AR are N-adj (post). Decision token: after the article/carrier.
- **clause**: canonical constituent order. EN/FR/ES/DE/AR/ZH are SVO; HI is SOV. **DE is restricted to simple main clauses** (no subordinate clauses, no fronted adjuncts) to avoid V2 inversion that polluted the Exp22 DE result.
- **adp**: adposition order. EN/FR/ES/DE/AR are prepositional; HI is postpositional. ZH has partial postposition (location + 里/上) — included cautiously.

**Donor design (same as Exp22, now with scene identity preserved).** For item `(scene s, language L)`, cross donor = scene `s` realized in a language `D` whose canonical order is opposite to `L`'s on that sub-phenomenon. Null = scene `s' ≠ s` in `L` (same canonical order). Metric: `frac_flip = (patched_margin − clean_margin) · sign(flip) / (2 · mean |clean_margin|)` as in Exp22. Not floor-corrected: read as "is the recipient's order decision pushable toward the donor's order."

**Representation.** For every (scene, language) we capture the residual at the decision position across layers. `R_retr`: for each layer, is the matched scene in another language the nearest neighbour after per-language centring, chance-corrected? Also CKA (Lim et al. Eq. 7) across language pairs for comparison.

**Known caveats (flag in paper, verify before publishing).**
- AR / HI / ZH items are hand-written by a non-native. Mark cells with `# VERIFY` and have Anwoy / a native speaker check before any public result.
- ZH adposition items use the postpositional location pattern (NP 里/上/下); this is less clear-cut than HI postposition. If ZH adp looks weird, drop ZH from adp only.
- Small n per sub-phenomenon per language (10–15). Report bootstrap CIs.

Untested on GPU in my sandbox; item structure and analysis were tested on a mock tokenizer and synthetic data.

In [1]:
# CELL 1 — config
import os
GPU_ID = 2
os.environ.setdefault("CUDA_VISIBLE_DEVICES", str(GPU_ID))
for v in ("HTTP_PROXY","HTTPS_PROXY","ALL_PROXY","http_proxy","https_proxy","all_proxy"): os.environ.pop(v, None)
os.environ["HF_HUB_OFFLINE"] = "1"

import json, warnings, time, gc, itertools, random
from pathlib import Path
import numpy as np

MODELS = {
    "bloom-1b7": "bigscience/bloom-1b7",
    "bloom-7b1": "bigscience/bloom-7b1",
    # add Gemma paths here when you extend
}
OUT = Path("exp25_results"); OUT.mkdir(exist_ok=True)
KINDS = ("resid", "attn", "mlp")
LAYER_STRIDE = 2
SEED = 0
LANGS = ("en", "fr", "es", "de", "ar", "hi", "zh")

# which langs are "pre" (adj first, SVO, pre-position) for each sub-phenomenon
CANONICAL = {
    "adj":    {"en":"pre","de":"pre","zh":"pre","hi":"pre","fr":"post","es":"post","ar":"post"},
    "clause": {"en":"pre","fr":"pre","es":"pre","de":"pre","ar":"pre","zh":"pre","hi":"post"},  # pre = SVO
    "adp":    {"en":"pre","fr":"pre","es":"pre","de":"pre","ar":"pre","hi":"post","zh":"post"},
}
def opposite_donor(sub, L):
    want = "post" if CANONICAL[sub][L] == "pre" else "pre"
    return [l for l in LANGS if CANONICAL[sub].get(l) == want]
print("config ok")

config ok


In [2]:
# CELL 2 — parallel items
# Each scene is a dict: {lang: (prefix, cand_canonical, cand_anti)}.
# `prefix` is the shared text up to the divergence; `cand_canonical` is the correct continuation
# in that language; `cand_anti` is the opposite-order (ungrammatical/marked) continuation.
# Candidates include their leading space/token boundary where needed.

PARALLEL_ADJ = [
  # 1 red house
  {"en":("The"," red house"," house red"),
   "fr":("La"," maison rouge"," rouge maison"),
   "es":("La"," casa roja"," roja casa"),
   "de":("Das"," rote Haus"," Haus rot"),
   "ar":("هذا ","البيت الأحمر","الأحمر البيت"),  # VERIFY
   "hi":("यह"," लाल घर है"," घर लाल है"),         # VERIFY
   "zh":("这是","红房子","房子红")},               # VERIFY
  # 2 big dog
  {"en":("A"," big dog"," dog big"),
   "fr":("Un"," chien grand"," grand chien"),
   "es":("Un"," perro grande"," grande perro"),
   "de":("Ein"," großer Hund"," Hund groß"),
   "ar":("هذا ","الكلب الكبير","الكبير الكلب"),   # VERIFY
   "hi":("यह"," बड़ा कुत्ता है"," कुत्ता बड़ा है"),  # VERIFY
   "zh":("这是","大狗","狗大")},                   # VERIFY
  # 3 new book
  {"en":("The"," new book"," book new"),
   "fr":("Le"," livre nouveau"," nouveau livre"),  # NB: in FR "nouveau" often precedes; this item may be weak — keep for n
   "es":("El"," libro nuevo"," nuevo libro"),
   "de":("Das"," neue Buch"," Buch neu"),
   "ar":("هذا ","الكتاب الجديد","الجديد الكتاب"),  # VERIFY
   "hi":("यह"," नई किताब है"," किताब नई है"),      # VERIFY
   "zh":("这是","新书","书新")},                    # VERIFY
  # 4 small cat
  {"en":("A"," small cat"," cat small"),
   "fr":("Un"," chat petit"," petit chat"),        # NB: "petit" normally precedes in FR
   "es":("Un"," gato pequeño"," pequeño gato"),
   "de":("Eine"," kleine Katze"," Katze klein"),
   "ar":("هذه ","القطة الصغيرة","الصغيرة القطة"),   # VERIFY
   "hi":("यह"," छोटी बिल्ली है"," बिल्ली छोटी है"),  # VERIFY
   "zh":("这是","小猫","猫小")},                     # VERIFY
  # 5 black car
  {"en":("A"," black car"," car black"),
   "fr":("Une"," voiture noire"," noire voiture"),
   "es":("Un"," coche negro"," negro coche"),
   "de":("Ein"," schwarzes Auto"," Auto schwarz"),
   "ar":("هذه ","السيارة السوداء","السوداء السيارة"),  # VERIFY
   "hi":("यह"," काली गाड़ी है"," गाड़ी काली है"),       # VERIFY
   "zh":("这是","黑车","车黑")},                        # VERIFY
  # 6 green tree
  {"en":("A"," green tree"," tree green"),
   "fr":("Un"," arbre vert"," vert arbre"),
   "es":("Un"," árbol verde"," verde árbol"),
   "de":("Ein"," grüner Baum"," Baum grün"),
   "ar":("هذه ","الشجرة الخضراء","الخضراء الشجرة"),   # VERIFY
   "hi":("यह"," हरा पेड़ है"," पेड़ हरा है"),           # VERIFY
   "zh":("这是","绿树","树绿")},                        # VERIFY
  # 7 old man
  {"en":("An"," old man"," man old"),
   "fr":("Un"," homme vieux"," vieux homme"),      # vieux often precedes in FR
   "es":("Un"," hombre viejo"," viejo hombre"),
   "de":("Ein"," alter Mann"," Mann alt"),
   "ar":("هذا ","الرجل العجوز","العجوز الرجل"),     # VERIFY
   "hi":("यह"," बूढ़ा आदमी है"," आदमी बूढ़ा है"),      # VERIFY
   "zh":("这是","老人","人老")},                      # VERIFY
  # 8 white bird
  {"en":("A"," white bird"," bird white"),
   "fr":("Un"," oiseau blanc"," blanc oiseau"),
   "es":("Un"," pájaro blanco"," blanco pájaro"),
   "de":("Ein"," weißer Vogel"," Vogel weiß"),
   "ar":("هذا ","الطائر الأبيض","الأبيض الطائر"),   # VERIFY
   "hi":("यह"," सफेद चिड़िया है"," चिड़िया सफेद है"),  # VERIFY
   "zh":("这是","白鸟","鸟白")},                      # VERIFY
  # 9 young girl
  {"en":("A"," young girl"," girl young"),
   "fr":("Une"," fille jeune"," jeune fille"),     # jeune typically precedes
   "es":("Una"," chica joven"," joven chica"),
   "de":("Ein"," junges Mädchen"," Mädchen jung"),
   "ar":("هذه ","الفتاة الصغيرة","الصغيرة الفتاة"), # VERIFY
   "hi":("यह"," जवान लड़की है"," लड़की जवान है"),     # VERIFY
   "zh":("这是","年轻女孩","女孩年轻")},               # VERIFY
  # 10 cold water
  {"en":("The"," cold water"," water cold"),
   "fr":("L'"," eau froide"," froide eau"),
   "es":("El"," agua fría"," fría agua"),
   "de":("Das"," kalte Wasser"," Wasser kalt"),
   "ar":("هذا ","الماء البارد","البارد الماء"),      # VERIFY
   "hi":("यह"," ठंडा पानी है"," पानी ठंडा है"),       # VERIFY
   "zh":("这是","冷水","水冷")},                       # VERIFY
]

# Clause order: simple SVO vs SOV. All items are simple main clauses.
# EN/FR/ES/DE/AR/ZH canonical SVO (cand_canonical = SVO; cand_anti = SOV).
# HI canonical SOV (cand_canonical = SOV; cand_anti = SVO).
# Decision: after subject, generate verb (SVO) or object (SOV).
PARALLEL_CLAUSE = [
  # 1 John sees the dog
  {"en":("John"," sees the dog"," the dog sees"),
   "fr":("Jean"," voit le chien"," le chien voit"),
   "es":("Juan"," ve al perro"," al perro ve"),
   "de":("Peter"," sieht den Hund"," den Hund sieht"),  # main clause, no V2 trigger
   "ar":("أحمد"," يرى الكلب"," الكلب يرى"),             # VERIFY
   "hi":("जॉन"," कुत्ते को देखता है"," देखता है कुत्ते को"),  # SOV canonical / SVO anti VERIFY
   "zh":("小明","看狗","狗看")},                          # VERIFY
  # 2 Mary reads a book
  {"en":("Mary"," reads a book"," a book reads"),
   "fr":("Marie"," lit un livre"," un livre lit"),
   "es":("María"," lee un libro"," un libro lee"),
   "de":("Anna"," liest ein Buch"," ein Buch liest"),
   "ar":("مريم"," تقرأ كتاباً"," كتاباً تقرأ"),         # VERIFY
   "hi":("मेरी"," एक किताब पढ़ती है"," पढ़ती है एक किताब"), # VERIFY
   "zh":("小丽","读书","书读")},                         # VERIFY
  # 3 The boy eats the apple
  {"en":("The boy"," eats the apple"," the apple eats"),
   "fr":("Le garçon"," mange la pomme"," la pomme mange"),
   "es":("El niño"," come la manzana"," la manzana come"),
   "de":("Der Junge"," isst den Apfel"," den Apfel isst"),
   "ar":("الولد"," يأكل التفاحة"," التفاحة يأكل"),       # VERIFY
   "hi":("लड़का"," सेब खाता है"," खाता है सेब"),           # VERIFY
   "zh":("男孩","吃苹果","苹果吃")},                      # VERIFY
  # 4 The woman buys bread
  {"en":("The woman"," buys bread"," bread buys"),
   "fr":("La femme"," achète du pain"," du pain achète"),
   "es":("La mujer"," compra pan"," pan compra"),
   "de":("Die Frau"," kauft Brot"," Brot kauft"),
   "ar":("المرأة"," تشتري الخبز"," الخبز تشتري"),        # VERIFY
   "hi":("औरत"," रोटी खरीदती है"," खरीदती है रोटी"),      # VERIFY
   "zh":("女人","买面包","面包买")},                       # VERIFY
  # 5 The student writes the essay
  {"en":("The student"," writes the essay"," the essay writes"),
   "fr":("L'étudiant"," écrit l'essai"," l'essai écrit"),
   "es":("El estudiante"," escribe el ensayo"," el ensayo escribe"),
   "de":("Der Student"," schreibt den Aufsatz"," den Aufsatz schreibt"),
   "ar":("الطالب"," يكتب المقال"," المقال يكتب"),         # VERIFY
   "hi":("छात्र"," निबंध लिखता है"," लिखता है निबंध"),      # VERIFY
   "zh":("学生","写文章","文章写")},                        # VERIFY
  # 6 The teacher helps the child
  {"en":("The teacher"," helps the child"," the child helps"),
   "fr":("Le professeur"," aide l'enfant"," l'enfant aide"),
   "es":("El maestro"," ayuda al niño"," al niño ayuda"),
   "de":("Der Lehrer"," hilft dem Kind"," dem Kind hilft"),
   "ar":("المعلم"," يساعد الطفل"," الطفل يساعد"),         # VERIFY
   "hi":("शिक्षक"," बच्चे की मदद करता है"," मदद करता है बच्चे की"), # VERIFY
   "zh":("老师","帮助孩子","孩子帮助")},                    # VERIFY
  # 7 The dog chases the cat
  {"en":("The dog"," chases the cat"," the cat chases"),
   "fr":("Le chien"," poursuit le chat"," le chat poursuit"),
   "es":("El perro"," persigue al gato"," al gato persigue"),
   "de":("Der Hund"," jagt die Katze"," die Katze jagt"),
   "ar":("الكلب"," يطارد القطة"," القطة يطارد"),          # VERIFY
   "hi":("कुत्ता"," बिल्ली का पीछा करता है"," पीछा करता है बिल्ली का"), # VERIFY
   "zh":("狗","追猫","猫追")},                             # VERIFY
  # 8 The man opens the door
  {"en":("The man"," opens the door"," the door opens"),
   "fr":("L'homme"," ouvre la porte"," la porte ouvre"),
   "es":("El hombre"," abre la puerta"," la puerta abre"),
   "de":("Der Mann"," öffnet die Tür"," die Tür öffnet"),
   "ar":("الرجل"," يفتح الباب"," الباب يفتح"),             # VERIFY
   "hi":("आदमी"," दरवाज़ा खोलता है"," खोलता है दरवाज़ा"),     # VERIFY
   "zh":("男人","开门","门开")},                             # VERIFY
]

# Adposition: pre vs post. 'pre' = adposition + NP (English "in the house"); 'post' = NP + adposition.
# Decision: at the start of the adposition phrase, after a shared carrier.
# ZH: use location + 里/上/下 pattern for 'post' effect; its canonical here is treated as post with cautious note.
PARALLEL_ADP = [
  # 1 in the house
  {"en":("He is",     " in the house",   " the house in"),
   "fr":("Il est",    " dans la maison", " la maison dans"),
   "es":("Está",      " en la casa",     " la casa en"),
   "de":("Er ist",    " im Haus",        " Haus im"),
   "ar":("هو",         " في البيت",       " البيت في"),         # VERIFY
   "hi":("वह"," घर में है"," में घर है"),                       # post canonical, pre anti VERIFY
   "zh":("他",         "在房子里",         "里房子在")},          # VERIFY — ZH is awkward here
  # 2 on the table
  {"en":("The book is"," on the table"," the table on"),
   "fr":("Le livre est"," sur la table"," la table sur"),
   "es":("El libro está"," sobre la mesa"," la mesa sobre"),
   "de":("Das Buch ist"," auf dem Tisch"," dem Tisch auf"),
   "ar":("الكتاب"," على الطاولة"," الطاولة على"),              # VERIFY
   "hi":("किताब"," मेज़ पर है"," पर मेज़ है"),                     # VERIFY
   "zh":("书",         "在桌子上",         "上桌子在")},          # VERIFY
  # 3 under the tree
  {"en":("The dog is"," under the tree"," the tree under"),
   "fr":("Le chien est"," sous l'arbre"," l'arbre sous"),
   "es":("El perro está"," bajo el árbol"," el árbol bajo"),
   "de":("Der Hund ist"," unter dem Baum"," dem Baum unter"),
   "ar":("الكلب"," تحت الشجرة"," الشجرة تحت"),                 # VERIFY
   "hi":("कुत्ता"," पेड़ के नीचे है"," नीचे पेड़ के है"),           # VERIFY
   "zh":("狗",         "在树下",           "下树在")},            # VERIFY
  # 4 near the river
  {"en":("The city is"," near the river"," the river near"),
   "fr":("La ville est"," près du fleuve"," du fleuve près"),
   "es":("La ciudad está"," cerca del río"," del río cerca"),
   "de":("Die Stadt liegt"," nahe dem Fluss"," dem Fluss nahe"),
   "ar":("المدينة"," قرب النهر"," النهر قرب"),                  # VERIFY
   "hi":("शहर"," नदी के पास है"," पास नदी के है"),                # VERIFY
   "zh":("城市",       "在河边",           "边河在")},             # VERIFY
  # 5 behind the wall
  {"en":("He stood"," behind the wall"," the wall behind"),
   "fr":("Il se tenait"," derrière le mur"," le mur derrière"),
   "es":("Estaba"," detrás del muro"," del muro detrás"),
   "de":("Er stand"," hinter der Mauer"," der Mauer hinter"),
   "ar":("وقف"," خلف الجدار"," الجدار خلف"),                     # VERIFY
   "hi":("वह"," दीवार के पीछे खड़ा था"," पीछे दीवार के खड़ा था"),    # VERIFY
   "zh":("他",         "在墙后",           "后墙在")},              # VERIFY
  # 6 inside the box
  {"en":("The key is"," inside the box"," the box inside"),
   "fr":("La clé est"," dans la boîte"," la boîte dans"),
   "es":("La llave está"," dentro de la caja"," de la caja dentro"),
   "de":("Der Schlüssel ist"," in der Kiste"," der Kiste in"),
   "ar":("المفتاح"," داخل الصندوق"," الصندوق داخل"),             # VERIFY
   "hi":("चाबी"," डिब्बे के अंदर है"," अंदर डिब्बे के है"),           # VERIFY
   "zh":("钥匙",       "在盒子里",         "里盒子在")},             # VERIFY
]

PARALLEL = {"adj": PARALLEL_ADJ, "clause": PARALLEL_CLAUSE, "adp": PARALLEL_ADP}

# normalise: a prefix ending in a space tokenizes differently from prefix+cont, so new_tokens() returns None and the item is skipped.
# Move the space to the front of both candidates.
for _sub in PARALLEL.values():
    for _sc in _sub:
        for _l, (_pr, _c0, _c1) in list(_sc.items()):
            if _pr.endswith(" "):
                _sc[_l] = (_pr.rstrip(), " " + _c0.lstrip(), " " + _c1.lstrip())

def enc(tok, text): return tok(text, add_special_tokens=False)["input_ids"]
def new_tokens(tok, prefix, cont):
    a, b = enc(tok, prefix), enc(tok, prefix + cont)
    return b[len(a):] if b[:len(a)] == a else None

def item_valid(tok, scene, lang):
    if lang not in scene: return False
    pr, c0, c1 = scene[lang]
    if not pr or c0 == c1: return False
    t0, t1 = new_tokens(tok, pr, c0), new_tokens(tok, pr, c1)
    return bool(t0) and bool(t1) and t0 != t1

def valid_items(tok, sub, lang):
    """Returns list of (scene_idx, prefix, cand_canonical, cand_anti) for scenes where (lang, donor) both parse."""
    scenes = PARALLEL[sub]; out = []
    for i, sc in enumerate(scenes):
        if not item_valid(tok, sc, lang): continue
        out.append(i)
    return out

print("items:",
    "adj", len(PARALLEL_ADJ),
    "clause", len(PARALLEL_CLAUSE),
    "adp", len(PARALLEL_ADP))

items: adj 10 clause 8 adp 6


In [3]:
# CELL 3 — analysis helpers (compat, frac_flip, R_retr, CKA)
FLOOR_MIN_FRAC = 0.30

def order_transfer(Pc, Pn, m_clean, y, boots=300, seed=0):
    """Pc: patched margins with opposite-order cross donor; Pn: with same-order null donor from a different scene.
       y: label such that correct candidate is cand_y (cand0=canonical, cand1=anti); so y=0 for pre-order langs, y=1 for post."""
    sgn = (1 - 2 * y)[:, None]
    denom = 2 * np.abs(m_clean).mean()
    Tc = (Pc - m_clean[:, None]) * sgn   # positive if donor pushes toward its order
    Tn = (Pn - m_clean[:, None]) * (-sgn)
    rng, n, bs = np.random.default_rng(seed), len(y), []
    for _ in range(boots):
        ix = rng.integers(0, n, n)
        bs.append(Tc[ix].mean(0) / (2 * np.abs(m_clean[ix]).mean() + 1e-9))
    lo, hi = np.percentile(np.array(bs), [2.5, 97.5], axis=0)
    return dict(frac_flip=Tc.mean(0)/denom, lo=lo, hi=hi, null=Tn.mean(0)/denom,
                flip_rate=((Pc * sgn) > 0).mean(0))

def retrieval(HA, HL):
    """Label-free: does the anchor's aligned scene sit nearest to this scene after per-language centring?
       HA[i,l,:] = anchor-language state of scene i at layer l; HL same, recipient language. Returns per-layer R_retr."""
    n, nl, rows = HL.shape[0], HL.shape[1], []
    for l in range(nl):
        A = HA[:, l] - HA[:, l].mean(0)
        B = HL[:, l] - HL[:, l].mean(0)
        # DEGENERATE GUARD: if the decision-position state is (nearly) identical across scenes (e.g. prefix is just "The"/"这是"),
        # centred vectors are ~0, all similarities tie at 0, and rank==1 is returned by default -> fake R_retr = 1.0.
        deg = max(float((np.linalg.norm(A, axis=1) < 1e-3).mean()), float((np.linalg.norm(B, axis=1) < 1e-3).mean()))
        if deg > 0.5:
            rows.append(dict(layer=l, top1=np.nan, R_retr=np.nan, mrr=np.nan)); continue
        A = A / (np.linalg.norm(A, axis=1, keepdims=True) + 1e-9)
        B = B / (np.linalg.norm(B, axis=1, keepdims=True) + 1e-9)
        S = B @ A.T
        rank = (S > np.diag(S)[:, None]).sum(1) + 1
        top1 = float((rank == 1).mean())
        rows.append(dict(layer=l, top1=top1,
                         R_retr=(top1 - 1/n) / (1 - 1/n) if n > 1 else np.nan,
                         mrr=float((1/rank).mean())))
    return rows

def cka_linear(X, Y):
    """Linear CKA (Lim et al. Eq. 7). X, Y: [n, d]."""
    X = X - X.mean(0, keepdims=True); Y = Y - Y.mean(0, keepdims=True)
    xtx = X.T @ X; yty = Y.T @ Y; xty = X.T @ Y
    num = (xty ** 2).sum()
    den = np.sqrt((xtx ** 2).sum() * (yty ** 2).sum()) + 1e-12
    return float(num / den)

print("analysis functions ok")

analysis functions ok


In [4]:
# CELL 4 — run (GPU): capture states + patch opposite-order and null donors, per sub-phenomenon, per lang
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
dev = "cuda" if torch.cuda.is_available() else "cpu"

def get_modules(model):
    if hasattr(model, "transformer") and hasattr(model.transformer, "h"):
        bl = list(model.transformer.h)
        return dict(resid=bl, attn=[b.self_attention.dense for b in bl],
                    mlp=[b.mlp.dense_4h_to_h for b in bl],
                    ln_f=model.transformer.ln_f, lm_head=model.lm_head)
    bl = list(model.model.layers)
    return dict(resid=bl, attn=[b.self_attn.o_proj for b in bl],
                mlp=[b.mlp.down_proj for b in bl],
                ln_f=model.model.norm, lm_head=model.lm_head)

def patch_hook(p, vec):
    def hook(mod, inp, out):
        tup = isinstance(out, tuple); o = (out[0] if tup else out).clone()
        o[:, p] = vec.to(o.dtype)
        return (o,) + tuple(out[1:]) if tup else o
    return hook

@torch.no_grad()
def margin(model, mods, tok, bos, prefix, c0t, c1t, patch=None):
    ids = bos + enc(tok, prefix); p = len(ids) - 1
    rows = [ids + c0t, ids + c1t]; T = max(map(len, rows))
    pad = tok.pad_token_id if tok.pad_token_id is not None else 0
    inp = torch.tensor([r + [pad] * (T - len(r)) for r in rows], device=dev)
    att = torch.tensor([[1] * len(r) + [0] * (T - len(r)) for r in rows], device=dev)
    h = mods[patch[0]][patch[1]].register_forward_hook(patch_hook(p, patch[2])) if patch else None
    try: logits = model(input_ids=inp, attention_mask=att).logits
    finally:
        if h: h.remove()
    lp = []
    for r, ct in enumerate((c0t, c1t)):
        g = torch.log_softmax(logits[r, p:p + len(ct)].float(), -1)
        lp.append(sum(g[j, t].item() for j, t in enumerate(ct)))
    return lp[1] - lp[0]

@torch.no_grad()
def capture(model, mods, tok, bos, prefix, kinds=KINDS):
    ids = bos + enc(tok, prefix); p = len(ids) - 1
    nl = len(mods["resid"])
    store = {k: [None] * nl for k in kinds}; hs = []
    for k in kinds:
        for li, m in enumerate(mods[k]):
            def hk(mod, inp, out, k=k, li=li):
                o = out[0] if isinstance(out, tuple) else out
                store[k][li] = o[0, p].detach()
            hs.append(m.register_forward_hook(hk))
    try: model(input_ids=torch.tensor([ids], device=dev))
    finally:
        for h in hs: h.remove()
    return {k: torch.stack(v) for k, v in store.items()}

def prepend_bos(key): return "bloom" not in key  # BLOOM already includes BOS via its tokenizer

for key, path in MODELS.items():
    tok = AutoTokenizer.from_pretrained(path)
    bos = [tok.bos_token_id] if prepend_bos(key) else []
    model = AutoModelForCausalLM.from_pretrained(path, torch_dtype=torch.bfloat16).to(dev).eval()
    mods = get_modules(model)
    nl = len(mods["resid"]); H = mods["lm_head"].weight.shape[1]
    LAYERS = sorted(set(list(range(0, nl, LAYER_STRIDE)) + [nl - 1]))

    for sub in ("adj", "clause", "adp"):
        for L in LANGS:
            f = OUT / f"{key}__{sub}__{L}.npz"
            if f.exists(): print("cached", f.name); continue
            scene_ids = valid_items(tok, sub, L)
            donors_for_L = opposite_donor(sub, L)
            if not scene_ids or not donors_for_L:
                print("SKIP", key, sub, L, "scenes=", len(scene_ids), "donors=", donors_for_L); continue
            # pick a primary donor for this (sub, L): first opposite-order lang with the most shared scenes
            donor_scores = []
            for D in donors_for_L:
                shared = [i for i in scene_ids if item_valid(tok, PARALLEL[sub][i], D)]
                donor_scores.append((D, shared))
            donor_scores.sort(key=lambda x: -len(x[1]))
            D, usable = donor_scores[0]
            if len(usable) < 4:
                print("SKIP", key, sub, L, "usable_scenes_with_donor=", len(usable), "donor=", D); continue
            scene_ids = usable

            t0 = time.time(); n = len(scene_ids)
            # label: c0 is canonical for EVERY language by construction of PARALLEL items, so y=0 for all.
            y = np.zeros(n, int)
            mc = np.zeros(n)
            P = {(d, k): np.zeros((n, len(LAYERS))) for d in ("cross", "null") for k in KINDS}
            HL = np.zeros((n, nl, H), dtype=np.float32)
            HA = np.zeros_like(HL)
            cache = {}
            def cap(prefix):
                if prefix not in cache: cache[prefix] = capture(model, mods, tok, bos, prefix)
                return cache[prefix]

            for i, s in enumerate(scene_ids):
                sc = PARALLEL[sub][s]
                pr_L, c0_L, c1_L = sc[L]
                pr_D, c0_D, c1_D = sc[D]
                c0t = new_tokens(tok, pr_L, c0_L); c1t = new_tokens(tok, pr_L, c1_L)
                mc[i] = margin(model, mods, tok, bos, pr_L, c0t, c1t)
                # cross donor state: same scene, opposite-order language, at ITS decision position (last token of prefix)
                dc = cap(pr_D)
                # null donor state: SAME language, a DIFFERENT scene with same canonical order (recipient's own order)
                other = [sj for sj in scene_ids if sj != s][i % max(1, n - 1)]
                dn = cap(PARALLEL[sub][other][L][0])
                for name, dcap in (("cross", dc), ("null", dn)):
                    for k in KINDS:
                        for j, l in enumerate(LAYERS):
                            P[(name, k)][i, j] = margin(model, mods, tok, bos, pr_L, c0t, c1t,
                                                        patch=(k, l, dcap[k][l]))
                HL[i] = cap(pr_L)["resid"].float().cpu().numpy()
                HA[i] = cap(pr_D)["resid"].float().cpu().numpy()
            np.savez(f, y=y, m_clean=mc, layers=np.array(LAYERS), scene_ids=np.array(scene_ids),
                     donor=np.array(D), HL=HL, HA=HA,
                     **{f"P_{d}_{k}": P[(d, k)] for d in ("cross", "null") for k in KINDS})
            acc = float(((mc > 0).astype(int) == y).mean())
            print(f"saved {f.name} n={n} donor={D} clean_acc={acc:.2f} {time.time()-t0:.0f}s")
    del model; gc.collect(); torch.cuda.empty_cache()

/home/anwoy/miniconda3/envs/ke/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 293/293 [00:00<00:00, 2297.69it/s]


cached bloom-1b7__adj__en.npz
cached bloom-1b7__adj__fr.npz
cached bloom-1b7__adj__es.npz
cached bloom-1b7__adj__de.npz
cached bloom-1b7__adj__ar.npz
cached bloom-1b7__adj__hi.npz
cached bloom-1b7__adj__zh.npz
cached bloom-1b7__clause__en.npz
cached bloom-1b7__clause__fr.npz
cached bloom-1b7__clause__es.npz
cached bloom-1b7__clause__de.npz
cached bloom-1b7__clause__ar.npz
cached bloom-1b7__clause__hi.npz
cached bloom-1b7__clause__zh.npz
cached bloom-1b7__adp__en.npz
cached bloom-1b7__adp__fr.npz
cached bloom-1b7__adp__es.npz
cached bloom-1b7__adp__de.npz
cached bloom-1b7__adp__ar.npz
cached bloom-1b7__adp__hi.npz
cached bloom-1b7__adp__zh.npz


Loading weights: 100%|██████████| 365/365 [00:00<00:00, 959.18it/s] 


cached bloom-7b1__adj__en.npz
cached bloom-7b1__adj__fr.npz
cached bloom-7b1__adj__es.npz
cached bloom-7b1__adj__de.npz
cached bloom-7b1__adj__ar.npz
cached bloom-7b1__adj__hi.npz
cached bloom-7b1__adj__zh.npz
cached bloom-7b1__clause__en.npz
cached bloom-7b1__clause__fr.npz
cached bloom-7b1__clause__es.npz
cached bloom-7b1__clause__de.npz
cached bloom-7b1__clause__ar.npz
cached bloom-7b1__clause__hi.npz
cached bloom-7b1__clause__zh.npz
cached bloom-7b1__adp__en.npz
cached bloom-7b1__adp__fr.npz
cached bloom-7b1__adp__es.npz
cached bloom-7b1__adp__de.npz
cached bloom-7b1__adp__ar.npz
cached bloom-7b1__adp__hi.npz
cached bloom-7b1__adp__zh.npz


In [5]:
# CELL 5 — analysis: per (sub, lang) frac_flip + R_retr, plus cross-language CKA; combined 2x2 verdict
import pandas as pd

rows_wo = []
R_by_pair = {}      # (model, sub, L, D) -> {layer: R_retr}
H_states = {}       # (model, sub, L) -> HL array (for cross-lang CKA across all L pairs)

for f in sorted(OUT.glob("*.npz")):
    key, sub, L = f.stem.split("__")
    z = np.load(f, allow_pickle=True)
    y, mc, layers = z["y"], z["m_clean"], z["layers"]; D = str(z["donor"])
    HL, HA = z["HL"], z["HA"]
    H_states[(key, sub, L)] = HL
    for kind in KINDS:
        ot = order_transfer(z[f"P_cross_{kind}"], z[f"P_null_{kind}"], mc, y)
        for j, l in enumerate(layers):
            rows_wo.append(dict(model=key, sub=sub, lang=L, donor=D, kind=kind, layer=int(l),
                                n=len(y), frac_flip=ot["frac_flip"][j], lo=ot["lo"][j], hi=ot["hi"][j],
                                null=ot["null"][j], flip_rate=ot["flip_rate"][j]))
    rt = retrieval(HA, HL)
    for r in rt:
        R_by_pair.setdefault((key, sub, L, D), {})[r["layer"]] = r["R_retr"]

df = pd.DataFrame(rows_wo)
df.to_csv(OUT / "wordorder_parallel_per_layer.csv", index=False)

# ---- per-cell late-layer summary (resid)
sw = df[df.kind == "resid"]
late = sw[sw.layer >= sw.layer.max() * 0.5].groupby(["model","sub","lang"]).agg(
    n=("n","first"),
    donor=("donor","first"),
    frac_flip_late=("frac_flip","median"),
    frac_flip_max=("frac_flip","max"),
    null_late=("null","median"),
).round(2)
print("\n=== Word-order patching (resid, late-layer median) ===\n", late.to_string())

# ---- R_retr summary per (model, sub, lang) -- median over late layers
rr = []
for (key, sub, L, D), per_layer in R_by_pair.items():
    layers_sorted = sorted(per_layer); late_layers = [l for l in layers_sorted if l >= max(layers_sorted) * 0.5]
    vals = [per_layer[l] for l in late_layers]
    rr.append(dict(model=key, sub=sub, lang=L, donor=D,
                   R_retr_late=round(float(np.nanmedian(vals)), 2),
                   R_retr_max=round(float(np.nanmax(vals)), 2)))
df_r = pd.DataFrame(rr).sort_values(["model","sub","lang"])
df_r.to_csv(OUT / "wordorder_parallel_R_retr.csv", index=False)
print("\n=== R_retr vs opposite-order donor (median over late layers) ===\n", df_r.to_string(index=False))

# ---- cross-language CKA at the decision token (all lang pairs, averaged over mid+late layers)
cka_rows = []
for key in {k for (k, s, l) in H_states}:
    for sub in {s for (k, s, l) in H_states if k == key}:
        langs_here = sorted({l for (k, s, l) in H_states if k == key and s == sub})
        for La, Lb in itertools.combinations(langs_here, 2):
            Ha = H_states[(key, sub, La)]; Hb = H_states[(key, sub, Lb)]
            # keep only scenes present in both (align by index since items are parallel-indexed)
            m = min(len(Ha), len(Hb))
            if m < 4: continue
            Ha, Hb = Ha[:m], Hb[:m]
            cks = [cka_linear(Ha[:, l], Hb[:, l]) for l in range(Ha.shape[1])]
            mid_late = cks[len(cks) // 3:]
            cka_rows.append(dict(model=key, sub=sub, lang_a=La, lang_b=Lb,
                                 cka_mid_late=round(float(np.mean(mid_late)), 2),
                                 cka_max=round(float(np.max(cks)), 2)))
df_cka = pd.DataFrame(cka_rows)
df_cka.to_csv(OUT / "wordorder_parallel_cka.csv", index=False)
if len(df_cka):
    print("\n=== CKA across language pairs (mean of mid+late layers) ===")
    for key in sorted(df_cka.model.unique()):
        for sub in sorted(df_cka[df_cka.model == key]["sub"].unique()):
            piv = df_cka[(df_cka.model == key) & (df_cka["sub"] == sub)].pivot(index="lang_a", columns="lang_b", values="cka_mid_late")
            print(f"\n[{key}] [{sub}]\n", piv.to_string())

# ---- 2x2 verdict per (model, sub, lang)
# comp_shared if frac_flip_late >= 0.30; rep_shared if R_retr_late >= 0.30 (bars for both are discussion-level, flag them)
j = late.reset_index().merge(df_r, on=["model","sub","lang","donor"], how="outer")
j["comp_shared"] = (j["frac_flip_late"] >= 0.30)          # positive transfer only; negative = not transfer
def _cell(r):
    comp = "shared" if r["frac_flip_late"] >= 0.30 else "sep"
    rep = "n/a" if (r["sub"] == "adj" or not np.isfinite(r["R_retr_late"])) else ("shared" if r["R_retr_late"] >= 0.30 else "sep")
    return f"{comp} comp / {rep} rep"          # adj R_retr is invalid (decision token precedes the scene), see notes
j["cell"] = j.apply(_cell, axis=1)
print("\n=== 2x2 verdict per cell (thresholds: |frac_flip|≥0.30, R_retr≥0.30 — tune) ===")
print(j[["model","sub","lang","donor","frac_flip_late","R_retr_late","cell"]].to_string(index=False))
j.to_csv(OUT / "wordorder_parallel_2x2.csv", index=False)


=== Word-order patching (resid, late-layer median) ===
                         n donor  frac_flip_late  frac_flip_max  null_late
model     sub    lang                                                    
bloom-1b7 adj    ar    10    en            0.25           0.26       0.03
                 de    10    fr           -0.18          -0.17      -0.05
                 en    10    fr            0.07           0.08      -0.05
                 es    10    en            0.19           0.20      -0.07
                 fr    10    en            0.16           0.16      -0.10
                 hi    10    fr            0.26           0.27       0.00
                 zh    10    fr            0.26           0.27       0.00
          adp    ar     6    hi            0.38           0.43      -0.01
                 de     6    hi           -0.06          -0.05      -0.07
                 en     6    hi           -0.25          -0.25      -0.06
                 es     6    hi            0.13        

/tmp/ipykernel_2339145/752111373.py:44: RuntimeWarning: All-NaN slice encountered
  R_retr_late=round(float(np.nanmedian(vals)), 2),
/tmp/ipykernel_2339145/752111373.py:45: RuntimeWarning: All-NaN axis encountered
  R_retr_max=round(float(np.nanmax(vals)), 2)))



=== CKA across language pairs (mean of mid+late layers) ===

[bloom-1b7] [adj]
 lang_b    de    en    es    fr   hi   zh
lang_a                                  
ar      0.29  0.39  0.15  0.11  0.0  0.0
de       NaN  0.91  0.69  0.36  0.0  0.0
en       NaN   NaN  0.61  0.31  0.0  0.0
es       NaN   NaN   NaN  0.56  0.0  0.0
fr       NaN   NaN   NaN   NaN  0.0  0.0
hi       NaN   NaN   NaN   NaN  NaN  0.0

[bloom-1b7] [adp]
 lang_b    de    en    es    fr    hi    zh
lang_a                                    
ar      0.38  0.40  0.56  0.45  0.19  0.23
de       NaN  0.88  0.52  0.87  0.44  0.50
en       NaN   NaN  0.57  0.97  0.43  0.73
es       NaN   NaN   NaN  0.57  0.54  0.67
fr       NaN   NaN   NaN   NaN  0.44  0.74
hi       NaN   NaN   NaN   NaN   NaN  0.90

[bloom-1b7] [clause]
 lang_b    de    en    es    fr    hi    zh
lang_a                                    
ar      0.16  0.18  0.17  0.17  0.17  0.15
de       NaN  0.98  0.97  0.97  0.16  0.95
en       NaN   NaN  0.99  0.99  